In [1]:
from pathlib import Path
import time
import gc

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

from lightgbm import LGBMClassifier

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
REPORTS_DIR = PROJECT_ROOT / "reports"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data: {DATA_DIR}")
print(f"Interim data: {INTERIM_DIR}")

Project root: c:\Projects\creditlens-ai
Raw data: c:\Projects\creditlens-ai\data\raw
Interim data: c:\Projects\creditlens-ai\data\interim


In [2]:
feature_files = {
    "bureau": INTERIM_DIR / "bureau_customer_features.csv",
    "previous": INTERIM_DIR / "previous_application_customer_features.csv",
    "installments": INTERIM_DIR / "installments_customer_features.csv",
}

for name, path in feature_files.items():
    print(
        f"{name:15} "
        f"exists={path.exists()} "
        f"path={path}"
    )

bureau          exists=True path=c:\Projects\creditlens-ai\data\interim\bureau_customer_features.csv
previous        exists=True path=c:\Projects\creditlens-ai\data\interim\previous_application_customer_features.csv
installments    exists=True path=c:\Projects\creditlens-ai\data\interim\installments_customer_features.csv


In [3]:
application_train = pd.read_csv(
    DATA_DIR / "application_train.csv",
    low_memory=False
)

bureau_features = pd.read_csv(
    feature_files["bureau"]
)

previous_features = pd.read_csv(
    feature_files["previous"]
)

installment_features = pd.read_csv(
    feature_files["installments"]
)

print(f"Application:  {application_train.shape}")
print(f"Bureau:       {bureau_features.shape}")
print(f"Previous:     {previous_features.shape}")
print(f"Installments: {installment_features.shape}")

Application:  (307511, 122)
Bureau:       (305811, 20)
Previous:     (338857, 24)
Installments: (339587, 30)


In [4]:
train_full = (
    application_train
    .merge(
        bureau_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )
    .merge(
        previous_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )
    .merge(
        installment_features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )
)

print(f"Full dataset shape: {train_full.shape}")
print(f"Duplicate customers: {train_full['SK_ID_CURR'].duplicated().sum()}")
print(f"Missing TARGET: {train_full['TARGET'].isna().sum()}")

Full dataset shape: (307511, 193)
Duplicate customers: 0
Missing TARGET: 0


In [5]:
train_full["BUREAU_HAS_HISTORY"] = (
    train_full["BUREAU_LOAN_COUNT"]
    .notna()
    .astype("int8")
)

train_full["PREV_HAS_HISTORY"] = (
    train_full["PREV_APPLICATION_COUNT"]
    .notna()
    .astype("int8")
)

train_full["INST_HAS_HISTORY"] = (
    train_full["INST_PAYMENT_RECORD_COUNT"]
    .notna()
    .astype("int8")
)

train_full["DAYS_EMPLOYED_ANOMALY"] = (
    train_full["DAYS_EMPLOYED"] == 365243
).astype("int8")

train_full.loc[
    train_full["DAYS_EMPLOYED"] == 365243,
    "DAYS_EMPLOYED"
] = np.nan

print(f"Dataset shape after engineered flags: {train_full.shape}")

Dataset shape after engineered flags: (307511, 197)


C:\Users\Vıctus\AppData\Local\Temp\ipykernel_3444\4129258819.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_full["BUREAU_HAS_HISTORY"] = (
C:\Users\Vıctus\AppData\Local\Temp\ipykernel_3444\4129258819.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_full["PREV_HAS_HISTORY"] = (
C:\Users\Vıctus\AppData\Local\Temp\ipykernel_3444\4129258819.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all 

In [6]:
y = train_full["TARGET"].copy()

X = train_full.drop(
    columns=[
        "TARGET",
        "SK_ID_CURR"
    ]
).copy()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Positive class rate: {y.mean() * 100:.2f}%")

X shape: (307511, 195)
y shape: (307511,)
Positive class rate: 8.07%


In [7]:
categorical_columns = (
    X
    .select_dtypes(
        include=["object", "string", "category"]
    )
    .columns
    .tolist()
)

numeric_columns = (
    X
    .select_dtypes(include=["number"])
    .columns
    .tolist()
)

for column in categorical_columns:
    X[column] = X[column].astype("category")

print(f"Numeric features: {len(numeric_columns)}")
print(f"Categorical features: {len(categorical_columns)}")
print(
    f"Total features: "
    f"{len(numeric_columns) + len(categorical_columns)}"
)

X[categorical_columns].dtypes.head(10)

Numeric features: 179
Categorical features: 16
Total features: 195


NAME_CONTRACT_TYPE     category
CODE_GENDER            category
FLAG_OWN_CAR           category
FLAG_OWN_REALTY        category
NAME_TYPE_SUITE        category
NAME_INCOME_TYPE       category
NAME_EDUCATION_TYPE    category
NAME_FAMILY_STATUS     category
NAME_HOUSING_TYPE      category
OCCUPATION_TYPE        category
dtype: object

In [8]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Train shape: {X_train.shape}")
print(f"Validation shape: {X_val.shape}")

print(
    f"Train positive rate: "
    f"{y_train.mean() * 100:.2f}%"
)

print(
    f"Validation positive rate: "
    f"{y_val.mean() * 100:.2f}%"
)

Train shape: (246008, 195)
Validation shape: (61503, 195)
Train positive rate: 8.07%
Validation positive rate: 8.07%


In [9]:
import lightgbm as lgb

from lightgbm import LGBMClassifier

print(f"LightGBM version: {lgb.__version__}")

LightGBM version: 4.7.0


In [10]:
lightgbm_model = LGBMClassifier(
    objective="binary",

    n_estimators=2000,
    learning_rate=0.03,

    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,

    subsample=0.8,
    colsample_bytree=0.8,

    reg_alpha=0.1,
    reg_lambda=0.1,

    class_weight="balanced",

    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lightgbm_model

,learning_rate,0.03
,n_estimators,2000
,objective,'binary'
,class_weight,'balanced'
,min_child_samples,50
,subsample,0.8
,colsample_bytree,0.8
,reg_alpha,0.1
,reg_lambda,0.1
,random_state,42
,n_jobs,-1


In [11]:
import time

start_time = time.time()

lightgbm_model.fit(
    X_train,
    y_train,

    eval_set=[
        (X_val, y_val)
    ],

    eval_metric="auc",

    categorical_feature=categorical_columns,

    callbacks=[
        lgb.early_stopping(
            stopping_rounds=100,
            verbose=True
        ),
        lgb.log_evaluation(period=100)
    ]
)

lightgbm_training_time = time.time() - start_time

print(
    f"Training time: "
    f"{lightgbm_training_time:.2f} seconds"
)

print(
    f"Best iteration: "
    f"{lightgbm_model.best_iteration_}"
)

print(
    f"Best validation score: "
    f"{lightgbm_model.best_score_}"
)

c:\Projects\creditlens-ai\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.763565	valid_0's binary_logloss: 0.57656
[200]	valid_0's auc: 0.775124	valid_0's binary_logloss: 0.552597
[300]	valid_0's auc: 0.778997	valid_0's binary_logloss: 0.539857
[400]	valid_0's auc: 0.78026	valid_0's binary_logloss: 0.531006
[500]	valid_0's auc: 0.780958	valid_0's binary_logloss: 0.523362
[600]	valid_0's auc: 0.781385	valid_0's binary_logloss: 0.516238
Early stopping, best iteration is:
[580]	valid_0's auc: 0.781471	valid_0's binary_logloss: 0.51758
Training time: 13.20 seconds
Best iteration: 580
Best validation score: defaultdict(<class 'collections.OrderedDict'>, {'valid_0': OrderedDict({'auc': np.float64(0.7814707729656786), 'binary_logloss': np.float64(0.5175804531124462)})})


In [12]:
lightgbm_probabilities = lightgbm_model.predict_proba(
    X_val
)[:, 1]

lightgbm_roc_auc = roc_auc_score(
    y_val,
    lightgbm_probabilities
)

lightgbm_pr_auc = average_precision_score(
    y_val,
    lightgbm_probabilities
)

print(
    f"LightGBM ROC-AUC: "
    f"{lightgbm_roc_auc:.6f}"
)

print(
    f"LightGBM PR-AUC: "
    f"{lightgbm_pr_auc:.6f}"
)

LightGBM ROC-AUC: 0.781471
LightGBM PR-AUC: 0.277886


In [13]:
lightgbm_predictions = (
    lightgbm_probabilities >= 0.50
).astype(int)

lightgbm_metrics = {
    "accuracy": accuracy_score(
        y_val,
        lightgbm_predictions
    ),
    "precision": precision_score(
        y_val,
        lightgbm_predictions,
        zero_division=0
    ),
    "recall": recall_score(
        y_val,
        lightgbm_predictions,
        zero_division=0
    ),
    "f1": f1_score(
        y_val,
        lightgbm_predictions,
        zero_division=0
    ),
    "roc_auc": lightgbm_roc_auc,
    "pr_auc": lightgbm_pr_auc
}

pd.Series(lightgbm_metrics)

accuracy     0.742956
precision    0.190596
recall       0.672709
f1           0.297034
roc_auc      0.781471
pr_auc       0.277886
dtype: float64

In [14]:
lightgbm_confusion_matrix = confusion_matrix(
    y_val,
    lightgbm_predictions
)

lightgbm_confusion_matrix

array([[42354, 14184],
       [ 1625,  3340]])

In [15]:
logistic_relational_metrics = {
    "accuracy": 0.702649,
    "precision": 0.170272,
    "recall": 0.692850,
    "f1": 0.273363,
    "roc_auc": 0.765474,
    "pr_auc": 0.247268
}

boosting_comparison = pd.DataFrame({
    "LogisticRegression": logistic_relational_metrics,
    "LightGBM": lightgbm_metrics
}).T

boosting_comparison

,accuracy,precision,recall,f1,roc_auc,pr_auc
LogisticRegression,0.702649,0.170272,0.692850,0.273363,0.765474,0.247268
LightGBM,0.742956,0.190596,0.672709,0.297034,0.781471,0.277886


In [16]:
threshold_results = []

for threshold in np.arange(0.10, 0.91, 0.01):

    predictions = (
        lightgbm_probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val,
        predictions
    ).ravel()

    threshold_results.append({
        "threshold": threshold,
        "accuracy": accuracy_score(
            y_val,
            predictions
        ),
        "precision": precision_score(
            y_val,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_val,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            y_val,
            predictions,
            zero_division=0
        ),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    })

threshold_results = pd.DataFrame(
    threshold_results
)

threshold_results.head()

,threshold,accuracy,precision,recall,f1,TN,FP,FN,TP
0,0.10,0.149830,0.086243,0.993353,0.158708,4283,52255,33,4932
1,0.11,0.166691,0.087572,0.989728,0.160906,5338,51200,51,4914
2,0.12,0.183373,0.088994,0.986908,0.163265,6378,50160,65,4900
3,0.13,0.201307,0.090525,0.983082,0.165784,7500,49038,84,4881
4,0.14,0.220396,0.092246,0.979255,0.168609,8693,47845,103,4862


In [17]:
best_f1_result = (
    threshold_results
    .sort_values(
        "f1",
        ascending=False
    )
    .head(10)
)

best_f1_result

,threshold,accuracy,precision,recall,f1,TN,FP,FN,TP
55,0.65,0.852820,0.261969,0.452971,0.331956,50202,6336,2716,2249
56,0.66,0.857958,0.267136,0.435650,0.331190,50604,5934,2802,2163
57,0.67,0.862917,0.272870,0.419335,0.330607,50990,5548,2883,2082
54,0.64,0.846658,0.255154,0.468681,0.330422,49745,6793,2638,2327
59,0.69,0.873486,0.287952,0.385096,0.329513,51810,4728,3053,1912
58,0.68,0.867763,0.278400,0.400806,0.328573,51380,5158,2975,1990
52,0.62,0.834951,0.244482,0.499698,0.328327,48871,7667,2484,2481
51,0.61,0.828675,0.239675,0.516616,0.327440,48401,8137,2400,2565
53,0.63,0.840284,0.248029,0.481571,0.327422,49289,7249,2574,2391
60,0.70,0.878640,0.296067,0.365358,0.327083,52225,4313,3151,1814


In [18]:
high_recall_thresholds = (
    threshold_results[
        threshold_results["recall"] >= 0.80
    ]
    .sort_values(
        "precision",
        ascending=False
    )
)

high_recall_thresholds.head(10)

,threshold,accuracy,precision,recall,f1,TN,FP,FN,TP
28,0.38,0.615596,0.150204,0.807654,0.253300,33851,22687,955,4010
27,0.37,0.602458,0.147049,0.817523,0.249263,32994,23544,906,4059
26,0.36,0.589467,0.144115,0.827190,0.245465,32147,24391,858,4107
25,0.35,0.575988,0.140996,0.835045,0.241257,31279,25259,819,4146
24,0.34,0.561696,0.138004,0.844310,0.237232,30354,26184,773,4192
23,0.33,0.547924,0.135265,0.852971,0.233501,29464,27074,730,4235
22,0.32,0.534023,0.132583,0.861027,0.229783,28569,27969,690,4275
21,0.31,0.520349,0.130174,0.869688,0.226453,27685,28853,647,4318
20,0.30,0.505309,0.127495,0.877543,0.222642,26721,29817,608,4357
19,0.29,0.490204,0.124762,0.883585,0.218650,25762,30776,578,4387


In [19]:
best_high_recall_threshold = (
    high_recall_thresholds
    .iloc[0]
)

best_high_recall_threshold

threshold        0.380000
accuracy         0.615596
precision        0.150204
recall           0.807654
f1               0.253300
TN           33851.000000
FP           22687.000000
FN             955.000000
TP            4010.000000
Name: 28, dtype: float64

In [21]:
selected_thresholds = pd.concat(
    [
        threshold_results[
            np.isclose(
                threshold_results["threshold"],
                threshold
            )
        ].iloc[[0]]
        for threshold in [0.38, 0.50, 0.65]
    ],
    ignore_index=True
)

selected_thresholds.insert(
    0,
    "strategy",
    [
        "High Recall",
        "Default",
        "Max F1"
    ]
)

selected_thresholds[
    [
        "strategy",
        "threshold",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "TN",
        "FP",
        "FN",
        "TP"
    ]
]

,strategy,threshold,accuracy,precision,recall,f1,TN,FP,FN,TP
0,High Recall,0.38,0.615596,0.150204,0.807654,0.253300,33851,22687,955,4010
1,Default,0.50,0.742956,0.190596,0.672709,0.297034,42354,14184,1625,3340
2,Max F1,0.65,0.852820,0.261969,0.452971,0.331956,50202,6336,2716,2249


In [22]:
cost_scenarios = []

for fn_cost_multiplier in [2, 5, 10, 20]:
    scenario = threshold_results.copy()

    scenario["cost"] = (
        scenario["FP"]
        + fn_cost_multiplier * scenario["FN"]
    )

    best_row = scenario.loc[
        scenario["cost"].idxmin()
    ]

    cost_scenarios.append({
        "FN_cost_multiplier": fn_cost_multiplier,
        "best_threshold": best_row["threshold"],
        "precision": best_row["precision"],
        "recall": best_row["recall"],
        "f1": best_row["f1"],
        "FP": int(best_row["FP"]),
        "FN": int(best_row["FN"]),
        "total_cost": best_row["cost"]
    })

cost_scenario_results = pd.DataFrame(cost_scenarios)

cost_scenario_results

,FN_cost_multiplier,best_threshold,precision,recall,f1,FP,FN,total_cost
0,2,0.82,0.443099,0.147432,0.221248,920,4233,9386.0
1,5,0.65,0.261969,0.452971,0.331956,6336,2716,19916.0
2,10,0.49,0.186754,0.687210,0.293695,14858,1553,30388.0
3,20,0.36,0.144115,0.827190,0.245465,24391,858,41551.0


In [23]:
threshold_report_path = (
    REPORTS_DIR / "lightgbm_threshold_analysis.csv"
)

threshold_results.to_csv(
    threshold_report_path,
    index=False
)

cost_report_path = (
    REPORTS_DIR / "lightgbm_cost_scenarios.csv"
)

cost_scenario_results.to_csv(
    cost_report_path,
    index=False
)

print(f"Saved: {threshold_report_path}")
print(f"Saved: {cost_report_path}")

Saved: c:\Projects\creditlens-ai\reports\lightgbm_threshold_analysis.csv
Saved: c:\Projects\creditlens-ai\reports\lightgbm_cost_scenarios.csv


In [24]:
boosting_comparison_path = (
    REPORTS_DIR / "boosting_model_comparison.csv"
)

boosting_comparison.to_csv(
    boosting_comparison_path,
    index=True
)

print(f"Saved: {boosting_comparison_path}")

Saved: c:\Projects\creditlens-ai\reports\boosting_model_comparison.csv


## LightGBM Model Result

A LightGBM classifier was trained using application-level and relational credit-history features.

### Model Performance

LightGBM improved predictive performance over the full relational Logistic Regression baseline:

| Model | ROC-AUC | PR-AUC | F1 @ 0.50 |
|---|---:|---:|---:|
| Logistic Regression | 0.7655 | 0.2473 | 0.2734 |
| LightGBM | 0.7815 | 0.2779 | 0.2970 |

The LightGBM model reached its best validation performance at iteration 580 using early stopping.

### Threshold Analysis

Classification performance varied substantially depending on the selected probability threshold.

- Threshold 0.38 produced approximately 80.8% recall.
- Threshold 0.50 produced approximately 67.3% recall.
- Threshold 0.65 maximized F1 at approximately 0.332.

Cost-sensitive experiments also demonstrated that the preferred threshold depends on the assumed relative cost of false negatives and false positives.

Therefore, no universal lending decision threshold is selected. Thresholds are treated as configurable operating scenarios for research and decision-support analysis rather than autonomous credit-approval rules.